In [ ]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
import os
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, ConfusionMatrixDisplay
)


In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────
LABELS_DIR   = '../dataset/labels/'
ORACLES_DIR  = '../dataset/labels/oracles/'
RESULTS_DIR  = 'results/'
CATEGORIES   = ['Offensivi', 'Steriotipi', 'Pregiudizi', 'Violenza', 'Niente']
CAT_NAMES_EN = ['Offensive', 'Stereotypes', 'Prejudices', 'Violence', 'Nothing']
ALL_LABELS   = list(range(5))  # ensures confusion matrix is always 5x5


In [ ]:
# ── Load data ───────────────────────────────────────────────────────────────
oracle_img   = pd.read_csv(ORACLES_DIR + 'oracle_multiclass_images.csv',  sep=';')
oracle_video = pd.read_csv(ORACLES_DIR + 'oracle_multiclass_videos.csv',  sep=';')
llm_img      = pd.read_csv(LABELS_DIR  + 'llm_multiclass_images.csv',     sep=';', header=[0,1])
llm_video    = pd.read_csv(LABELS_DIR  + 'llm_multiclass_videos.csv',     sep=';', header=[0,1])

# Flatten multi-level header: ('Offensivi', 'GPT4') -> 'Offensivi GPT4'
llm_img.columns   = [' '.join(c).strip() if isinstance(c, tuple) else c for c in llm_img.columns]
llm_video.columns = [' '.join(c).strip() if isinstance(c, tuple) else c for c in llm_video.columns]

# First column is the file path
llm_img.rename(columns={llm_img.columns[0]:   'Collegamento'}, inplace=True)
llm_video.rename(columns={llm_video.columns[0]: 'Collegamento'}, inplace=True)


In [ ]:
# ── Metrics helper ──────────────────────────────────────────────────────────
def save_metrics(y_true, y_pred, model_name, subset, average):
    """Compute multilabel metrics + per-class confusion matrix and save to disk."""
    y_true_idx = y_true.argmax(axis=1)
    y_pred_idx = y_pred.argmax(axis=1)

    acc   = accuracy_score(y_true, y_pred)
    prec  = precision_score(y_true, y_pred, average=average, zero_division=0)
    rec   = recall_score(y_true, y_pred, average=average, zero_division=0)
    f1    = f1_score(y_true, y_pred, average=average, zero_division=0)
    cm    = confusion_matrix(y_true_idx, y_pred_idx, labels=ALL_LABELS)
    report = classification_report(
        y_true_idx, y_pred_idx,
        labels=ALL_LABELS, target_names=CAT_NAMES_EN, zero_division=0
    )

    out_dir = os.path.join(RESULTS_DIR, subset, model_name)
    os.makedirs(out_dir, exist_ok=True)

    # Confusion matrix
    fig, ax = plt.subplots(figsize=(8, 8))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CAT_NAMES_EN)
    disp.plot(ax=ax, cmap=plt.cm.Greens, colorbar=True)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f'confusion_matrix_{average}.pdf'))
    plt.close()

    # Metrics text
    with open(os.path.join(out_dir, f'metrics_{average}.txt'), 'w') as fh:
        fh.write(report + '\n')
        fh.write(f'Accuracy:  {acc:.4f}\n')
        fh.write(f'Precision: {prec:.4f}\n')
        fh.write(f'Recall:    {rec:.4f}\n')
        fh.write(f'F1:        {f1:.4f}\n')

    print(f'[{model_name} | {subset} | {average}]  F1={f1:.3f}  Acc={acc:.3f}')


## Images — GPT-4, Claude Haiku, Claude Sonnet


In [ ]:
merged_img = pd.merge(
    llm_img, oracle_img,
    how='inner', left_on='Collegamento', right_on='Collegamento'
)

y_true_img = merged_img[CATEGORIES].values

gpt4_img   = merged_img[[f'{c} GPT4'   for c in CATEGORIES]].values
haiku_img  = merged_img[[f'{c} Haiku'  for c in CATEGORIES]].values
sonnet_img = merged_img[[f'{c} Sonnet' for c in CATEGORIES]].values

for avg in ['micro', 'macro', 'samples', 'weighted']:
    save_metrics(y_true_img, gpt4_img,   'GPT4',          'images', avg)
    save_metrics(y_true_img, haiku_img,  'Claude-Haiku',  'images', avg)
    save_metrics(y_true_img, sonnet_img, 'Claude-Sonnet', 'images', avg)


## Videos — Gemini 1.0 Pro Vision, 1.5 Flash, 1.5 Pro


In [ ]:
merged_video = pd.merge(
    llm_video, oracle_video,
    how='inner', left_on='Collegamento', right_on='Collegamento'
)

y_true_vid = merged_video[CATEGORIES].values

flash_vid      = merged_video[[f'{c} gemini-1.5-flash'      for c in CATEGORIES]].values
pro_vid        = merged_video[[f'{c} gemini-1.5-pro'        for c in CATEGORIES]].values
pro_vision_vid = merged_video[[f'{c} gemini-1.0-pro-vision' for c in CATEGORIES]].values

for avg in ['micro', 'macro', 'samples', 'weighted']:
    save_metrics(y_true_vid, flash_vid,      'Gemini-1.5-flash',      'videos', avg)
    save_metrics(y_true_vid, pro_vid,        'Gemini-1.5-pro',        'videos', avg)
    save_metrics(y_true_vid, pro_vision_vid, 'Gemini-1.0-pro-vision', 'videos', avg)
